Pruebas de validación de pepBERT 
1. Balanceo de datos
2. trabajo de pruebas del modelo preentrenado

In [16]:
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from dotenv import load_dotenv, find_dotenv

# ¡LA MAGIA ESTÁ AQUÍ! 
# find_dotenv() escanea hacia atrás (hacia la carpeta accelerated-drug-design) buscando el .env
load_dotenv(find_dotenv(), override=True) 

# Buscar la variable exacta
ruta_dataset = os.getenv('ruta_dataset2')

# --- FILTRO DE SEGURIDAD ---
if ruta_dataset is None:
    print("❌ ERROR: Aún no encuentro 'ruta_dataset2'.")
    print("Revisa que el archivo .env esté guardado y que la variable se llame exactamente igual.")
elif not os.path.exists(ruta_dataset):
    print(f"❌ ERROR: Encontré la ruta en el .env, pero el CSV no está físicamente en este lugar: {ruta_dataset}")
else:
    # Si pasa las pruebas, cargamos el dataset
    print("✅ ¡Ruta .env y archivo encontrados! Cargando el Dataset...", '----'*10)
    df = pd.read_csv(ruta_dataset)
    print(f"🎉 Dataset cargado exitosamente. Tamaño: {df.shape}")
    display(df.head(2)) # Muestra las primeras 2 filas para confirmar

✅ ¡Ruta .env y archivo encontrados! Cargando el Dataset... ----------------------------------------
🎉 Dataset cargado exitosamente. Tamaño: (28800, 8)


,mpnn,plddt,ptm,i_ptm,pae,i_pae,rmsd,seq
0,1.610,0.759,0.716,0.687,6.965,7.368,5.898,GYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTN...
1,1.593,0.589,0.664,0.493,10.559,11.163,1.549,GYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTN...


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from dotenv import load_dotenv

# 1. Le decimos a Python que el archivo está una carpeta atrás (..) y se llama rutas.env
ruta_archivo_env = os.path.join(os.getcwd(), "..", "rutas.env")

# 2. Forzamos a dotenv a leer ESE archivo específico
load_dotenv(dotenv_path=ruta_archivo_env, override=True)

# 3. Ahora sí, pedimos nuestra variable secreta
ruta_dataset = os.getenv('ruta_dataset2')

# --- FILTRO DE SEGURIDAD ---
if ruta_dataset is None:
    print(f"❌ ERROR: No pude encontrar la variable 'ruta_dataset2' dentro de tu archivo rutas.env")
    print(f"Asegúrate de que el archivo exista en: {ruta_archivo_env}")
elif not os.path.exists(ruta_dataset):
    print(f"❌ ERROR: La variable se cargó en secreto, pero el CSV no está físicamente en esa ruta.")
else:
    print("🔒 Leyendo configuración segura desde rutas.env...")
    print("✅ ¡Ruta secreta cargada con éxito!", '----'*10)
    df = pd.read_csv(ruta_dataset)
    print(f"🎉 Dataset cargado. Tamaño: {df.shape}")
    display(df.head(5))

🔒 Leyendo configuración segura desde rutas.env...
✅ ¡Ruta secreta cargada con éxito! ----------------------------------------
🎉 Dataset cargado. Tamaño: (28800, 8)


,mpnn,plddt,ptm,i_ptm,pae,i_pae,rmsd,seq
0,1.610,0.759,0.716,0.687,6.965,7.368,5.898,GYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTN...
1,1.593,0.589,0.664,0.493,10.559,11.163,1.549,GYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTN...


In [18]:
# --- CELDA 2: BALANCEO DE DATOS ---
def balancear_regresion_continua(df_input, target_col='i_ptm', umbral_asimetria=1.0):
    df_out = df_input.copy()
    sesgo = df_out[target_col].skew()
    
    print(f"📊 Analizando distribución de '{target_col}'... Skewness: {sesgo:.3f}")
    
    if abs(sesgo) <= umbral_asimetria:
        print("✅ Distribución equilibrada. Asignando peso neutral.")
        df_out['sample_weight'] = 1.0
        return df_out

    print("⚠️ Desbalanceo detectado. Aplicando pesos dinámicos...")
    p90, p95, p99 = df_out[target_col].quantile([0.90, 0.95, 0.99])
    
    condiciones = [
        df_out[target_col] >= p99,
        df_out[target_col] >= p95,
        df_out[target_col] >= p90
    ]
    pesos = [25.0, 10.0, 3.0]
    
    # Aplicar pesos, los datos normales se quedan con 1.0
    df_out['sample_weight'] = np.select(condiciones, pesos, default=1.0)
    return df_out

# Ejecutamos la función
df_balanceado = balancear_regresion_continua(df, target_col='i_ptm')

print("\n✅ Resumen de pesos asignados:")
print(df_balanceado['sample_weight'].value_counts().sort_index())

📊 Analizando distribución de 'i_ptm'... Skewness: 0.122
✅ Distribución equilibrada. Asignando peso neutral.

✅ Resumen de pesos asignados:
sample_weight
1.0    28800
Name: count, dtype: int64
